# SINDy-Autoencoder — Test-Set Results

Evaluates a trained `SINDyAutoencoderModule` on the held-out test snapshots:

1. **Autoencoder reconstruction** — encoder → decoder, compared against the true test fields (video, elastic-energy trajectory, reconstruction error).
2. **SINDy latent dynamics** — integrate the learned latent ODE `dz/dt = Θ(z)Ξ` from the encoded initial condition and compare against the encoded latent trajectory.
3. **SINDy-simulated reconstruction** — decode the *simulated* latent trajectory (not the encoded one) and compare the resulting fields against the true data.

All fields are visualized through `tr(C) = C00 + C11`, recovered from the model's native square-root representation `(B00, B01, B11)` via the closed-form relation `C00 = B00²+B01²`, `C11 = B11²+B01²`, `C01 = B01(B00+B11)` (the analytic inverse of the Newton solve used to build the dataset).

In [ ]:
import os
import sys
import glob

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# assumes the notebook's working directory is reduction_methods/sind_ae
SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
sys.path.append(SRC_DIR)
from Autoencoder.SINDy_Autoencoder import SINDyAutoencoderModule

In [ ]:
# --- paths ---------------------------------------------------------------
TEST_DIR = "C:/Users/fabio/Desktop/npz_data/snapshots_dataset/test"   # saved B_*.pt / dB_dt_*.pt test snapshots
CHECKPOINT_PATH = "best_sindy_ae_checkpoint.pt"                        # checkpoint produced by train_sindy_AE.py

# --- physical constants (must match the dataset the model was trained on) ---
Wi, Re, beta, L2 = 5.0, 1.0, 0.5, 20.0
dt = 0.001   # spacing between consecutive snapshots

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
bp = checkpoint["best_params"]

model = SINDyAutoencoderModule(
    bp["n_filters"], bp["n_layers"], checkpoint["latent_dim"],
    checkpoint["input_shape"], checkpoint["degree"], checkpoint["include_bias"],
).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"latent_dim={checkpoint['latent_dim']}  degree={checkpoint['degree']}  "
      f"n_filters={bp['n_filters']}  n_layers={bp['n_layers']}  rec_energy={checkpoint['rec_energy']}")

### Physics & data helpers

In [ ]:
def calc_energy(A, Re, Wi, beta, L2):
    """A: (H, W, T, 3) numpy array with components (C00, C01, C11). Returns elastic energy per snapshot."""
    def s(A, Wi, L2):
        traA = A[..., 0:1] + A[..., 2:3]
        I = np.ones_like(A)
        I[..., 0] = 0
        return -1 / Wi * (L2 / (L2 - traA) * A - I)
    t = -(1 - beta) * s(A, Wi, L2)
    traT = t[..., 0] + t[..., 2]
    return 1 / Re * traT.sum((0, 1))


def load_ordered_snapshots(folder):
    """Loads B_*.pt / dB_dt_*.pt in temporal order. Returns (states, derivs) as (T, 3, H, W) tensors."""
    files = sorted(glob.glob(os.path.join(folder, "B_*.pt")))
    states = torch.stack([torch.load(f).float() for f in files])
    derivs = torch.stack([
        torch.load(os.path.join(folder, f"dB_dt_{i:05d}.pt")).float()
        for i in range(len(files))
    ])
    return states, derivs


def B_to_A(B):
    """B: (T, 3, H, W) with channel order (B00, B01, B11).
    Closed-form inverse of the Newton square-root decomposition used to build the dataset.
    Returns (T, H, W, 3) as (C00, C01, C11)."""
    B00, B01, B11 = B[:, 0], B[:, 1], B[:, 2]
    A00 = B00 ** 2 + B01 ** 2
    A11 = B11 ** 2 + B01 ** 2
    A01 = B01 * (B00 + B11)
    return torch.stack([A00, A01, A11], dim=-1)


def derive_A_products(B_states):
    """B_states: (T, 3, H, W) -> (A shaped (H, W, T, 3) for calc_energy, trace tr(C) shaped (H, W, T)), both numpy."""
    A = B_to_A(B_states)
    A_energy_shape = A.permute(1, 2, 0, 3).numpy()
    trace = (A[..., 0] + A[..., 2]).permute(1, 2, 0).numpy()
    return A_energy_shape, trace


def relative_error_traj(true_states, rec_states):
    """Per-snapshot relative L2 error, ||true-rec|| / ||true||."""
    true_flat = true_states.flatten(1)
    rec_flat = rec_states.flatten(1)
    num = (true_flat - rec_flat).norm(dim=1)
    den = true_flat.norm(dim=1).clamp_min(1e-12)
    return (num / den).numpy()

### Model-evaluation helpers

In [ ]:
@torch.no_grad()
def chunked_encode_decode(model, states, batch_size=100):
    model.eval()
    latents, recons = [], []
    for i in range(0, len(states), batch_size):
        chunk = states[i:i + batch_size].to(device)
        z = model.encoder(chunk)
        x_hat = model.decoder(z)
        latents.append(z.cpu())
        recons.append(x_hat.cpu())
    return torch.cat(latents), torch.cat(recons)


@torch.no_grad()
def chunked_decode(model, latents, batch_size=100):
    model.eval()
    outs = []
    for i in range(0, len(latents), batch_size):
        outs.append(model.decoder(latents[i:i + batch_size].to(device)).cpu())
    return torch.cat(outs)


@torch.no_grad()
def rk4_integrate(sindy_module, z0, n_steps, dt):
    """Integrates dz/dt = sindy_module(z) forward from z0 for n_steps snapshots spaced by dt."""
    sindy_module.eval()
    z = z0.clone()
    traj = [z.clone()]
    for _ in range(n_steps - 1):
        k1 = sindy_module(z)
        k2 = sindy_module(z + 0.5 * dt * k1)
        k3 = sindy_module(z + 0.5 * dt * k2)
        k4 = sindy_module(z + dt * k3)
        z = z + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
        traj.append(z.clone())
    return torch.cat(traj, dim=0)

### Visualization helpers (video + static preview)

In [ ]:
def make_comparison_video(true_field, rec_field, times, save_path, title, fps=25, cmap="viridis"):
    """true_field, rec_field: (H, W, T) numpy arrays of a scalar field."""
    vmin = float(min(true_field.min(), rec_field.min()))
    vmax = float(max(true_field.max(), rec_field.max()))
    diff = np.abs(true_field - rec_field)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    im0 = axes[0].imshow(true_field[..., 0], vmin=vmin, vmax=vmax, cmap=cmap)
    im1 = axes[1].imshow(rec_field[..., 0], vmin=vmin, vmax=vmax, cmap=cmap)
    im2 = axes[2].imshow(diff[..., 0], cmap="inferno")
    for ax, label in zip(axes, ("True", "Reconstructed", "|Error|")):
        ax.set_title(label)
        ax.set_xticks([]); ax.set_yticks([])
    suptitle = fig.suptitle(f"{title}   t = {times[0]:.4f}")
    fig.tight_layout()

    def update(frame):
        im0.set_data(true_field[..., frame])
        im1.set_data(rec_field[..., frame])
        im2.set_data(diff[..., frame])
        suptitle.set_text(f"{title}   t = {times[frame]:.4f}")
        return im0, im1, im2, suptitle

    ani = animation.FuncAnimation(fig, update, frames=true_field.shape[-1], blit=False)
    try:
        ani.save(save_path, fps=fps, writer="ffmpeg")
    except (FileNotFoundError, ValueError):
        gif_path = os.path.splitext(save_path)[0] + ".gif"
        ani.save(gif_path, fps=fps, writer="pillow")
        print(f"ffmpeg not found - saved GIF instead: {gif_path}")
    plt.close(fig)
    return ani


def show_preview_frames(true_field, rec_field, times, n_preview=4, cmap="viridis"):
    """Static grid of a few sampled frames, as a quick sanity check without playing the video."""
    T = true_field.shape[-1]
    idxs = np.linspace(0, T - 1, n_preview).astype(int)
    vmin = float(min(true_field.min(), rec_field.min()))
    vmax = float(max(true_field.max(), rec_field.max()))
    fig, axes = plt.subplots(3, n_preview, figsize=(3 * n_preview, 8))
    row_labels = ("True", "Reconstructed", "|Error|")
    for col, idx in enumerate(idxs):
        frames = (true_field[..., idx], rec_field[..., idx], np.abs(true_field[..., idx] - rec_field[..., idx]))
        for row, frame in enumerate(frames):
            ax = axes[row, col]
            if row < 2:
                ax.imshow(frame, cmap=cmap, vmin=vmin, vmax=vmax)
            else:
                ax.imshow(frame, cmap="inferno")
            ax.set_xticks([]); ax.set_yticks([])
        axes[0, col].set_title(f"t = {times[idx]:.3f}")
    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(label)
    fig.tight_layout()
    return fig

## 1. Autoencoder reconstruction

In [ ]:
B_true_test, dB_true_test = load_ordered_snapshots(TEST_DIR)
T = B_true_test.shape[0]
t_axis = np.arange(T) * dt
print(f"Loaded {T} test snapshots of shape {tuple(B_true_test.shape[1:])}")

z_true_traj, B_rec_test = chunked_encode_decode(model, B_true_test)

A_true_energy, true_trace = derive_A_products(B_true_test.cpu())
A_rec_energy, rec_trace = derive_A_products(B_rec_test)

true_energy_traj = calc_energy(A_true_energy, Re, Wi, beta, L2)
rec_energy_traj = calc_energy(A_rec_energy, Re, Wi, beta, L2)
rec_error_traj = relative_error_traj(B_true_test.cpu(), B_rec_test)

In [ ]:
ae_video_path = "ae_reconstruction.mp4"
make_comparison_video(true_trace, rec_trace, t_axis, ae_video_path,
                       title="Autoencoder reconstruction — tr(C)")
print(f"Saved: {ae_video_path}")
show_preview_frames(true_trace, rec_trace, t_axis)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(t_axis, true_energy_traj, label="True")
axes[0].plot(t_axis, rec_energy_traj, "--", label="AE reconstruction")
axes[0].set_yscale("log")
axes[0].set_xlabel("t"); axes[0].set_ylabel("Elastic energy")
axes[0].set_title("Elastic energy"); axes[0].legend()

axes[1].plot(t_axis, rec_error_traj, color="tab:red")
axes[1].set_xlabel("t"); axes[1].set_ylabel("relative L2 error")
axes[1].set_title("AE reconstruction error")
fig.tight_layout()
plt.show()

## 2. SINDy latent dynamics

In [ ]:
z0 = z_true_traj[0:1].to(device)
z_sindy_traj = rk4_integrate(model.sindy, z0, n_steps=T, dt=dt).cpu()

latent_dim = z_true_traj.shape[1]
fig, axes = plt.subplots(latent_dim, 1, figsize=(9, 2 * latent_dim), sharex=True, squeeze=False)
axes = axes[:, 0]
for i, ax in enumerate(axes):
    ax.plot(t_axis, z_true_traj[:, i].numpy(), label="Encoded (true)")
    ax.plot(t_axis, z_sindy_traj[:, i].numpy(), "--", label="SINDy simulation")
    ax.set_ylabel(f"$z_{i}$")
axes[0].legend()
axes[-1].set_xlabel("t")
fig.suptitle("Latent trajectories: encoder vs. SINDy simulation")
fig.tight_layout()
plt.show()

## 3. SINDy-simulated reconstruction vs. true data

In [ ]:
B_sindy_rec = chunked_decode(model, z_sindy_traj)
A_sindy_energy, sindy_trace = derive_A_products(B_sindy_rec)
sindy_energy_traj = calc_energy(A_sindy_energy, Re, Wi, beta, L2)

sindy_video_path = "sindy_reconstruction.mp4"
make_comparison_video(true_trace, sindy_trace, t_axis, sindy_video_path,
                       title="SINDy-simulated reconstruction — tr(C)")
print(f"Saved: {sindy_video_path}")
show_preview_frames(true_trace, sindy_trace, t_axis)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t_axis, true_energy_traj, label="True")
ax.plot(t_axis, sindy_energy_traj, "--", label="SINDy simulation")
ax.set_yscale("log")
ax.set_xlabel("t"); ax.set_ylabel("Elastic energy")
ax.set_title("Elastic energy: true vs. SINDy-simulated reconstruction")
ax.legend()
fig.tight_layout()
plt.show()